In [ ]:
import pandas as pd
import os

from dotenv import load_dotenv
from snowflake.snowpark import Session
from sklearn.metrics import mean_absolute_error


load_dotenv()

conn = {
    "account": os.getenv("account"),
    "user": os.getenv("user"),
    "password": os.getenv("password"),
    "role": "ACCOUNTADMIN",
    "warehouse": "COMPUTE_WH",
    "database": "LH_NAUTICALS"
}

session = Session.builder.configs(conn).create()


df_snowpark = session.table("LH_NAUTICALS.QUESTAO7.VENDAS_PREVISAO")
df_vendas = df_snowpark.to_pandas()


df_vendas.columns = df_vendas.columns.str.strip().str.upper()

print("Colunas de vendas:")
print(df_vendas.columns)

print("\nHead vendas:")
print(df_vendas.head())


df_snowpark = session.table("LH_NAUTICALS.QUESTAO7.PRODUTOS_RAW")
df_produtos = df_snowpark.to_pandas()


df_produtos.columns = df_produtos.columns.str.strip().str.upper()

print("\nColunas de produtos:")
print(df_produtos.columns)

print("\nHead produtos:")
print(df_produtos.head())

Colunas de vendas:
Index(['ID', 'ID_CLIENT', 'ID_PRODUCT', 'QTD', 'TOTAL', 'SALE_DATE'], dtype='object')

Head vendas:
   ID  ID_CLIENT  ID_PRODUCT  QTD    TOTAL   SALE_DATE
0  id  id_client  id_product  qtd    total   sale_date
1   0         42         105   11   3405.0  2023-09-10
2   1          3         136    9  16873.9  15-09-2024
3   2         25         139    7   9475.3  2024-08-13
4   4         20          23    5  55893.0  2023-02-03

Colunas de produtos:
Index(['NAME', 'PRICE', 'CODE', 'ACTUAL_CATEGORY'], dtype='object')

Head produtos:
                             NAME        PRICE  CODE        ACTUAL_CATEGORY
0     Transponder AIS Maré Magnum  R$ 33122.52     1            ELETRONICOS
1       Transponder Furuno Marlin  R$ 13998.15     2            ELETRONICOS
2    Radar Furuno Pulse Leviathan   R$ 9024.19     3  E L E T R Ô N I C O S
3       Rádio AIS Hydro Tidal Zen   R$ 3381.88     4            Eletrunicos
4  Piloto Automático Furuno Storm  R$ 23669.01     5            E

In [3]:
## Ajustar tipos
df_vendas["ID_PRODUCT"] = pd.to_numeric(df_vendas["ID_PRODUCT"], errors="coerce")
df_vendas["QTD"] = pd.to_numeric(df_vendas["QTD"], errors="coerce")
df_vendas["TOTAL"] = pd.to_numeric(df_vendas["TOTAL"], errors="coerce")
df_vendas["SALE_DATE"] = pd.to_datetime(df_vendas["SALE_DATE"], dayfirst=True, errors="coerce")

df_produtos["CODE"] = pd.to_numeric(df_produtos["CODE"], errors="coerce")

C:\Users\Admin\AppData\Local\Temp\ipykernel_8176\3817824186.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_vendas["SALE_DATE"] = pd.to_datetime(df_vendas["SALE_DATE"], dayfirst=True, errors="coerce")


In [ ]:


produto_alvo = "Motor de Popa Yamaha Evo Dash 155HP"

produto_encontrado = df_produtos[
    df_produtos["NAME"].str.contains(produto_alvo, case=False, na=False)
].copy()

print("\nProduto encontrado:")
print(produto_encontrado)


if produto_encontrado.empty:
    raise ValueError("Produto alvo não encontrado na planilha de produtos.")

if len(produto_encontrado) > 1:
    print("\nAtenção: mais de um produto encontrado. Verifique manualmente:")
    print(produto_encontrado[["CODE", "NAME"]])


id_produto_alvo = produto_encontrado.iloc[0]["CODE"]

print(f"\nID_PRODUCT encontrado para o produto alvo: {id_produto_alvo}")



Produto encontrado:
                                   NAME         PRICE  CODE ACTUAL_CATEGORY
54  Motor de Popa Yamaha Evo Dash 155HP  R$ 121534.82    54      Propulssão

ID_PRODUCT encontrado para o produto alvo: 54


In [ ]:


df_produto = df_vendas[df_vendas["ID_PRODUCT"] == id_produto_alvo].copy()

print("\nQuantidade de linhas de vendas do produto alvo:")
print(len(df_produto))

print("\nAmostra das vendas do produto:")
print(df_produto.head())

df_produto = df_produto.dropna(subset=["SALE_DATE", "QTD"])

print("\nMenor data:", df_produto["SALE_DATE"].min())
print("Maior data:", df_produto["SALE_DATE"].max())



Quantidade de linhas de vendas do produto alvo:
62

Amostra das vendas do produto:
      ID ID_CLIENT  ID_PRODUCT   QTD       TOTAL  SALE_DATE
47    48        13        54.0  15.0  1823022.00 2024-05-30
54    55        35        54.0   3.0   346373.80 2024-11-24
72    74        45        54.0  11.0  1270038.85 2024-09-25
443  451        42        54.0  13.0  1500955.35 2024-02-19
495  503        45        54.0  11.0  1270038.85 2024-11-27

Menor data: 2023-01-10 00:00:00
Maior data: 2024-11-27 00:00:00


In [ ]:

df_diario = (
    df_produto.groupby("SALE_DATE", as_index=False)["QTD"]
    .sum()
    .sort_values("SALE_DATE")
)

df_diario = df_diario.set_index("SALE_DATE").asfreq("D", fill_value=0)
df_diario = df_diario.rename(columns={"QTD": "VENDAS"})

print("\nSérie diária:")
print(df_diario.head(10))
print(df_diario.tail(10))


Série diária:
            VENDAS
SALE_DATE         
2023-01-10     3.0
2023-01-11     0.0
2023-01-12     0.0
2023-01-13     0.0
2023-01-14     0.0
2023-01-15     0.0
2023-01-16     0.0
2023-01-17     0.0
2023-01-18     0.0
2023-01-19     0.0
            VENDAS
SALE_DATE         
2024-11-18     0.0
2024-11-19     0.0
2024-11-20     0.0
2024-11-21     0.0
2024-11-22     0.0
2024-11-23     0.0
2024-11-24    16.0
2024-11-25     0.0
2024-11-26     0.0
2024-11-27    23.0


In [ ]:

treino = df_diario.loc[: "2023-12-31"].copy()
teste = df_diario.loc["2024-01-01":"2024-01-31"].copy()

print("\nÚltima data treino:", treino.index.max())
print("Primeira data teste:", teste.index.min())
print("Última data teste:", teste.index.max())
print("Qtd dias teste:", len(teste))


if treino.empty:
    raise ValueError("O conjunto de treino ficou vazio. Verifique as datas do produto.")
if teste.empty:
    raise ValueError("O conjunto de teste ficou vazio. Verifique se existem dados para janeiro de 2024.")



Última data treino: 2023-12-31 00:00:00
Primeira data teste: 2024-01-01 00:00:00
Última data teste: 2024-01-31 00:00:00
Qtd dias teste: 31


In [ ]:

previsoes = []
historico = treino["VENDAS"].copy()

for data in teste.index:
    previsao_dia = historico.tail(7).mean()
    previsoes.append(previsao_dia)

    
    valor_real = teste.loc[data, "VENDAS"]
    historico.loc[data] = valor_real

teste["PREVISAO_BASELINE"] = previsoes


In [ ]:

mae = mean_absolute_error(teste["VENDAS"], teste["PREVISAO_BASELINE"])

print(f"\nMAE do baseline: {mae:.2f}")


resultado = teste.reset_index().rename(columns={
    "SALE_DATE": "DATA",
    "VENDAS": "REAL",
    "PREVISAO_BASELINE": "PREVISTO"
})

print("\nPrevisões diárias de janeiro de 2024:")
print(resultado)



MAE do baseline: 1.00

Previsões diárias de janeiro de 2024:
         DATA  REAL  PREVISTO
0  2024-01-01   0.0  0.000000
1  2024-01-02   0.0  0.000000
2  2024-01-03   0.0  0.000000
3  2024-01-04   0.0  0.000000
4  2024-01-05   0.0  0.000000
5  2024-01-06   0.0  0.000000
6  2024-01-07   0.0  0.000000
7  2024-01-08   0.0  0.000000
8  2024-01-09   0.0  0.000000
9  2024-01-10   0.0  0.000000
10 2024-01-11   0.0  0.000000
11 2024-01-12   0.0  0.000000
12 2024-01-13   0.0  0.000000
13 2024-01-14   0.0  0.000000
14 2024-01-15   0.0  0.000000
15 2024-01-16   0.0  0.000000
16 2024-01-17   0.0  0.000000
17 2024-01-18   0.0  0.000000
18 2024-01-19   0.0  0.000000
19 2024-01-20   0.0  0.000000
20 2024-01-21  11.0  0.000000
21 2024-01-22   6.0  1.571429
22 2024-01-23   0.0  2.428571
23 2024-01-24   0.0  2.428571
24 2024-01-25   0.0  2.428571
25 2024-01-26   0.0  2.428571
26 2024-01-27   0.0  2.428571
27 2024-01-28   0.0  2.428571
28 2024-01-29   0.0  0.857143
29 2024-01-30   0.0  0.000000
30 2024-

In [10]:
print("\nResposta objetiva:")
print("a. O baseline é adequado?")
print("   O baseline pode servir como referência inicial, mas sua adequação depende da estabilidade das vendas do produto.")
print("   Se houver muita oscilação diária, o modelo tende a apresentar erro mais alto.")

print("\nb. Limitação do método:")
print("   A média móvel de 7 dias não captura tendência, sazonalidade, promoções ou eventos externos.")


Resposta objetiva:
a. O baseline é adequado?
   O baseline pode servir como referência inicial, mas sua adequação depende da estabilidade das vendas do produto.
   Se houver muita oscilação diária, o modelo tende a apresentar erro mais alto.

b. Limitação do método:
   A média móvel de 7 dias não captura tendência, sazonalidade, promoções ou eventos externos.


In [11]:
primeira_semana = teste.loc["2024-01-01":"2024-01-07"]

soma_previsao = primeira_semana["PREVISAO_BASELINE"].sum()

soma_previsao_arredondada = round(soma_previsao)

print("\nSoma da previsão (01/01 a 07/01):", soma_previsao_arredondada)

print(primeira_semana[["VENDAS", "PREVISAO_BASELINE"]])



Soma da previsão (01/01 a 07/01): 0
            VENDAS  PREVISAO_BASELINE
SALE_DATE                            
2024-01-01     0.0                0.0
2024-01-02     0.0                0.0
2024-01-03     0.0                0.0
2024-01-04     0.0                0.0
2024-01-05     0.0                0.0
2024-01-06     0.0                0.0
2024-01-07     0.0                0.0
